ESERCIZIO

Filtro Classi e Cambio Algoritmo

Obbiettivo
Modificare lo script di tracking della lezione, per renderlo specifico per il monitoraggio pedonale e testare un algoritmo di associazione più robusto

Task
Filtro Classi: modifica la chiamata al metodo .track() affinchè il modello rlevi e segua solo le persone.

Suggerimento: nel dataset COCO, l'indice della classe 'person' è 0. Cerca il parametro classes nella documentazione di Ultralytics

Cambio algoritmo: sostituisci l'algoritmo di tracking da 'ByteTrack' a 'BoT-SORT'

Suggerimento: cambia il valore del paramtro tracker in 'botsor.yaml'

Analisi: osserva se il sistema diventa più preciso nel mantenere lo stesso ID quando una persona viene tempraneamente coperta da un altro oggetto.



In [1]:
import os

# --- CONFIGURAZIONE AMBIENTE  ---
# Keras 3 con backend Torch per la massima velocità di inferenza
os.environ["KERAS_BACKEND"] = "torch"

import cv2
import requests
import tempfile
from ultralytics import YOLO

class YOLOVideoTracker:
    """
    Pipeline per il tracking pedonale.  Utilizza l'algoritmo BoT-SORT e filtri per singola classe.
    """
    def __init__(self, model_variant='yolov8n.pt'):
        print(f"Inizializzazione YOLO con Tracking Nativo: {model_variant}...")
        self.model = YOLO(model_variant)

    def download_video(self, url):
        """Scarica il video in un file temporaneo per l'elaborazione OpenCV."""
        print(f"Scaricamento video: {url}")
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, stream=True)
        response.raise_for_status()
        
        temp_video = tempfile.NamedTemporaryFile(delete=False, suffix='.mp4')
        for chunk in response.iter_content(chunk_size=8192):
            temp_video.write(chunk)
        temp_video.close()
        return temp_video.name

    def process_video(self, video_url, output_path="output_person_botsort.mp4"):
        video_path = self.download_video(video_url)
        cap = cv2.VideoCapture(video_path)
        
        # Estrazione metadati video
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        
        # Configurazione Output
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        print("Tracking in corso: Filtro PERSONE e algoritmo BoT-SORT...")
        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret or frame_count > 100:
                break

            # 1. 'classes=[0]': Filtra l'inferenza solo sulla classe 'person' (COCO index 0).
            # 2. 'tracker="botsort.yaml"': Passa da ByteTrack a BoT-SORT per una gestione 
            #    delle occlusioni e del movimento camera più robusta.
            results = self.model.track(
                source=frame, 
                persist=True, 
                conf=0.3, 
                iou=0.5, 
                classes=[0],           # <--- Task 1: Filtro Persone
                tracker="botsort.yaml", # <--- Task 2: BoT-SORT
                verbose=False
            )

            # Renderizziamo solo i box delle persone rilevate e i loro ID
            annotated_frame = results[0].plot()

            # Scrittura del frame nel video finale
            out.write(annotated_frame)
            
            frame_count += 1
            if frame_count % 20 == 0:
                print(f"Processati {frame_count} frame...")

        cap.release()
        out.release()
        os.unlink(video_path)
        print(f"Tracking concluso! Video salvato in: {output_path}")

if __name__ == "__main__":
    video_processor = YOLOVideoTracker()
    
    # URL del video di esempio con persone e veicoli
    VIDEO_URL = "https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4"
    
    video_processor.process_video(VIDEO_URL)

Inizializzazione YOLO con Tracking Nativo: yolov8n.pt...
Scaricamento video: https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4
Tracking in corso: Filtro PERSONE e algoritmo BoT-SORT...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ----------------------------------- ---- 1.3/1.5 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 4.6 MB/s  0:00:00

[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

requirements: AutoUpdate success  4.3s
WARNING requirements: Restart runtime or rerun command for updates to take effect

Processati 20 frame...
Processati 40 frame...
Processati 60 frame...
Processati 80 frame...
Processati 100 frame...
Tracking concluso! Video salvato in: output_person_botsort.mp4
